In [ ]:
!pip install -q nltk scikit-learn beautifulsoup4 requests lxml pandas

In [ ]:
import re
import time
import nltk
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from nltk import pos_tag
from nltk.corpus import movie_reviews, stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 200)

for package in ['movie_reviews', 'punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4',
                'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(package, quiet=True)

In [ ]:
docs = [(movie_reviews.raw(file_id), 1 if category == 'pos' else 0)
        for category in movie_reviews.categories()
        for file_id in movie_reviews.fileids(category)]

df = pd.DataFrame(docs, columns=['text', 'label'])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df.shape)
print(df['label'].value_counts())
df.head()

In [ ]:
negations = {'no', 'nor', 'not', 'don', 'didn', 'doesn', 'isn', 'wasn', 'aren', 'weren',
             'won', 'wouldn', 'couldn', 'shouldn', 'hasn', 'haven', 'hadn'}
stop_words = set(stopwords.words('english')) - negations
lemmatizer = WordNetLemmatizer()


def wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    if tag.startswith('V'):
        return wordnet.VERB
    if tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN


def preprocess(text):
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text).lower()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if len(t) > 1 and t not in stop_words]
    tagged = pos_tag(tokens)
    lemmas = [lemmatizer.lemmatize(t, wordnet_pos(tag)) for t, tag in tagged]
    return ' '.join(lemmas)


print(df['text'][0][:300])
print(preprocess(df['text'][0])[:300])

In [ ]:
df['clean'] = df['text'].apply(preprocess)
df[['clean', 'label']].head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])

vectorizers = {
    'Bag of Words': CountVectorizer(max_features=10000, min_df=2, ngram_range=(1, 2)),
    'TF-IDF': TfidfVectorizer(max_features=10000, min_df=2, ngram_range=(1, 2)),
}

models = {
    'Naive Bayes': MultinomialNB(),
    'SVM': LinearSVC(random_state=42, max_iter=10000),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
}

results = []
fitted = {}

for vec_name, vectorizer in vectorizers.items():
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)
    for model_name, model in models.items():
        trained = clone(model).fit(X_train_vec, y_train)
        y_pred = trained.predict(X_test_vec)
        results.append({
            'Векторизация': vec_name,
            'Модель': model_name,
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': precision_score(y_test, y_pred),
            'Recall': recall_score(y_test, y_pred),
            'F1': f1_score(y_test, y_pred),
        })
        fitted[(vec_name, model_name)] = (vectorizer, trained, y_pred)

results_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
results_df

In [ ]:
best = results_df.iloc[0]
best_key = (best['Векторизация'], best['Модель'])
best_vectorizer, best_model, best_pred = fitted[best_key]

print('Лучшая модель:', best_key)
print(classification_report(y_test, best_pred, target_names=['negative', 'positive']))

ConfusionMatrixDisplay(confusion_matrix(y_test, best_pred),
                       display_labels=['negative', 'positive']).plot(cmap='Blues')
plt.title(f'Confusion matrix: {best_key[1]} + {best_key[0]}')
plt.show()

In [ ]:
urls = [
    'https://www.imdb.com/title/tt0111161/reviews/',
    'https://www.imdb.com/title/tt0468569/reviews/',
    'https://www.imdb.com/title/tt1375666/reviews/',
    'https://www.imdb.com/title/tt0368226/reviews/',
    'https://www.imdb.com/title/tt0185183/reviews/',
    'https://www.imdb.com/title/tt5697572/reviews/',
]

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/124.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
}

selectors = [
    'div.ipc-html-content-inner-div',
    'div.text.show-more__control',
    '[itemprop="reviewBody"]',
    'p.review-text',
    'span.review-text',
    'div.review-text',
]


def scrape(url):
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    nodes = []
    for selector in selectors:
        nodes = soup.select(selector)
        if nodes:
            break
    if not nodes:
        nodes = [p for p in soup.find_all('p') if len(p.get_text(strip=True)) > 80]
    texts = [node.get_text(' ', strip=True) for node in nodes]
    return [t for t in texts if len(t.split()) >= 15]


new_texts = []
for url in urls:
    try:
        found = scrape(url)
        print(url, '->', len(found))
        new_texts += found
    except Exception as error:
        print(url, '-> ошибка:', error)
    time.sleep(1)

new_texts = list(dict.fromkeys(new_texts))
if not new_texts:
    raise RuntimeError('Не удалось получить тексты с указанных сайтов')

new_df = pd.DataFrame({'text': new_texts})
new_df['clean'] = new_df['text'].apply(preprocess)
new_df = new_df[new_df['clean'].str.len() > 0].reset_index(drop=True)
print('Получено текстов:', len(new_df))

In [ ]:
X_new = best_vectorizer.transform(new_df['clean'])
new_df['prediction'] = best_model.predict(X_new)
new_df['sentiment'] = new_df['prediction'].map({1: 'positive', 0: 'negative'})

if hasattr(best_model, 'predict_proba'):
    new_df['confidence'] = best_model.predict_proba(X_new).max(axis=1).round(3)
else:
    new_df['confidence'] = np.abs(best_model.decision_function(X_new)).round(3)

new_df['review'] = new_df['text'].str.slice(0, 110) + '...'

print(new_df['sentiment'].value_counts())
new_df[['review', 'sentiment', 'confidence']]